# BioNutriGPT — Unified End-to-End Multimodal Transformer (No Liver Attributes)

This notebook trains **one unified model** end-to-end on the existing BioNutriGPT CSV.

### Architecture
**Biomarkers → Transformer → Z_bio**  
**Nutrition → Transformer → Z_nutri**  
**Lifestyle + Demographics → Transformer → Z_life**  
**Z_bio + Z_nutri + Z_life → Cross-Modal Transformer → Z_fused**  
**Z_fused → Adaptive Mixture of Experts → final Diabetes / Obesity / CVD heads**

There are **no liver-function attributes** in this version.

The three modality branches are components of **one PyTorch model**. There is one optimizer and one joint multitask loss. Gradients backpropagate from the final task heads through the MoE, cross-modal attention, and all three modality Transformers.

In [20]:
# 1. Install dependencies
!pip -q install scikit-learn joblib

In [21]:
# 2. Imports, reproducibility, and T4 device
import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [22]:
# 3. Configure paths
# Your CSV is uploaded directly into the Colab /content directory.

DATA_PATH = "/content/BioNutriGPT_final_dataset_2005_2018.csv"
OUTPUT_DIR = Path("/content/bionutrigpt_multimodal_no_liver")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("CSV exists:", Path(DATA_PATH).exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_PATH: /content/BioNutriGPT_final_dataset_2005_2018.csv
CSV exists: True
OUTPUT_DIR: /content/bionutrigpt_multimodal_no_liver


In [23]:
# 4. Load and inspect the dataset
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

TARGET_COLS = [
    "TARGET_DIABETES",
    "TARGET_OBESITY",
    "TARGET_CVD",
]

missing_targets = [c for c in TARGET_COLS if c not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

print("\nTargets found:")
for c in TARGET_COLS:
    print(f"  {c}: {df[c].notna().sum():,} non-missing")

Dataset shape: (42143, 839)
Number of columns: 839

Targets found:
  TARGET_DIABETES: 41,202 non-missing
  TARGET_OBESITY: 39,875 non-missing
  TARGET_CVD: 39,739 non-missing


## 5. Input feature groups — no liver attributes

The biomarker branch intentionally contains only the original 8 BioNutriGPT biomarkers:

- `LBXGH`
- `LBXGLU`
- `LBXIN`
- `LBDHDD`
- `LBXTR`
- `LBDLDL`
- `LBXAPB`
- `LBXTC`

No ALT, AST, GGT, albumin, bilirubin, alkaline phosphatase, or total protein columns are used.

In [24]:
# 5. Define feature groups

BIOMARKER_COLS = [
    "LBXGH",    # HbA1c
    "LBXGLU",   # Glucose
    "LBXIN",    # Insulin
    "LBDHDD",   # HDL
    "LBXTR",    # Triglycerides
    "LBDLDL",   # LDL
    "LBXAPB",   # ApoB
    "LBXTC",    # Total cholesterol
]

# Demographics/context explicitly used by the architecture.
DEMO_COLS = [
    "RIDAGEYR",
    "INDHHINC",
    "INDFMPIR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
]

# Nutrition: select available NHANES dietary/supplement variables.
# This avoids hard-coding variables that may not exist in the selected CSV.
NUTRITION_PREFIXES = (
    "DR1", "DR2", "DS1", "DS2"
)

# Lifestyle: physical activity, sleep, smoking and alcohol.
LIFESTYLE_PREFIXES = (
    "PAQ", "SLQ", "SMQ", "ALQ"
)

def available_exact(cols):
    return [c for c in cols if c in df.columns]

def available_prefixes(prefixes):
    return [
        c for c in df.columns
        if any(c.startswith(p) for p in prefixes)
    ]

BIOMARKER_COLS = available_exact(BIOMARKER_COLS)
DEMO_COLS = available_exact(DEMO_COLS)
NUTRITION_COLS = available_prefixes(NUTRITION_PREFIXES)
LIFESTYLE_COLS = available_prefixes(LIFESTYLE_PREFIXES)

# Exclude targets, IDs, and obvious outcome/body-size variables from input.
EXCLUDE_FROM_LIFESTYLE = set(TARGET_COLS) | {
    "SEQN",
    "BMXBMI", "BMXWT", "BMXHT", "BMXWAIST", "BMXARMC",
    "BMXTRI", "BMXTHICR", "BMXLEG", "BMXCALF",
}

LIFESTYLE_COLS = [
    c for c in LIFESTYLE_COLS
    if c not in EXCLUDE_FROM_LIFESTYLE
]

# Avoid duplicated demographic variables in the lifestyle/context branch.
LIFESTYLE_COLS = [c for c in LIFESTYLE_COLS if c not in DEMO_COLS]

print("Biomarkers:", len(BIOMARKER_COLS), BIOMARKER_COLS)
print("Nutrition:", len(NUTRITION_COLS))
print("Lifestyle:", len(LIFESTYLE_COLS))
print("Demographics:", len(DEMO_COLS))

if len(BIOMARKER_COLS) != 8:
    raise ValueError(
        "Expected all 8 original biomarkers. Missing: "
        + str([
            c for c in [
                "LBXGH","LBXGLU","LBXIN","LBDHDD",
                "LBXTR","LBDLDL","LBXAPB","LBXTC"
            ] if c not in df.columns
        ])
    )

if len(NUTRITION_COLS) == 0:
    raise ValueError("No nutrition columns matching DR1/DR2/DS1/DS2 were found.")

if len(LIFESTYLE_COLS) == 0:
    raise ValueError("No lifestyle columns matching PAQ/SLQ/SMQ/ALQ were found.")

Biomarkers: 8 ['LBXGH', 'LBXGLU', 'LBXIN', 'LBDHDD', 'LBXTR', 'LBDLDL', 'LBXAPB', 'LBXTC']
Nutrition: 83
Lifestyle: 184
Demographics: 7


In [25]:
# 6. Convert selected inputs and targets to numeric
INPUT_GROUPS = {
    "bio": BIOMARKER_COLS,
    "nutri": NUTRITION_COLS,
    "life": LIFESTYLE_COLS,
    "demo": DEMO_COLS,
}

for cols in INPUT_GROUPS.values():
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

for c in TARGET_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Numeric conversion complete.")

Numeric conversion complete.


## 7. Split without target leakage

We keep patients with partially missing target labels.

A separate mask is maintained for each target:
- mask = 1 → target contributes to the loss
- mask = 0 → target is missing and contributes no loss

This is preferable to dropping every patient who is missing any one of the three labels.

In [26]:
# 7. Patient split
# Stratify using diabetes where available; missing diabetes is assigned to an auxiliary class.

y_all = df[TARGET_COLS].to_numpy(dtype=np.float32)
split_strata = np.where(
    np.isnan(y_all[:, TARGET_COLS.index("TARGET_DIABETES")]),
    -1,
    y_all[:, TARGET_COLS.index("TARGET_DIABETES")]
).astype(int)

indices = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=split_strata
)

temp_strata = split_strata[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_strata
)

print("Train:", len(train_idx))
print("Val:  ", len(val_idx))
print("Test: ", len(test_idx))

Train: 29500
Val:   6321
Test:  6322


In [27]:
# 8. Preprocess each numeric modality.
# Fit imputers/scalers ONLY on the training split.

def fit_transform_group(cols):
    X = df[cols].to_numpy(dtype=np.float32)

    # IMPORTANT:
    # keep_empty_features=True prevents SimpleImputer from dropping
    # columns that are completely missing in the training split.
    # This keeps X and the missingness mask dimensionally aligned.
    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True
    )

    scaler = RobustScaler()

    X_train_raw = X[train_idx]
    X_val_raw = X[val_idx]
    X_test_raw = X[test_idx]

    # Fit ONLY on training data
    X_train = imputer.fit_transform(X_train_raw)
    X_val = imputer.transform(X_val_raw)
    X_test = imputer.transform(X_test_raw)

    # Scale using training statistics
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    # IMPORTANT:
    # Create masks BEFORE imputation.
    M_train = np.isnan(X_train_raw).astype(np.float32)
    M_val = np.isnan(X_val_raw).astype(np.float32)
    M_test = np.isnan(X_test_raw).astype(np.float32)

    return (
        X_train, X_val, X_test,
        M_train, M_val, M_test,
        imputer, scaler
    )


# ------------------------------------------------------------
# Process each modality
# ------------------------------------------------------------

bio = fit_transform_group(BIOMARKER_COLS)

nutri = fit_transform_group(NUTRITION_COLS)

life = fit_transform_group(LIFESTYLE_COLS)

demo = fit_transform_group(DEMO_COLS)


# ------------------------------------------------------------
# Unpack
# ------------------------------------------------------------

(
    X_bio_train, X_bio_val, X_bio_test,
    M_bio_train, M_bio_val, M_bio_test,
    bio_imp, bio_scaler
) = bio

(
    X_nutri_train, X_nutri_val, X_nutri_test,
    M_nutri_train, M_nutri_val, M_nutri_test,
    nutri_imp, nutri_scaler
) = nutri

(
    X_life_train, X_life_val, X_life_test,
    M_life_train, M_life_val, M_life_test,
    life_imp, life_scaler
) = life

(
    X_demo_train, X_demo_val, X_demo_test,
    M_demo_train, M_demo_val, M_demo_test,
    demo_imp, demo_scaler
) = demo


# ------------------------------------------------------------
# Verify dimensions
# ------------------------------------------------------------

print("Biomarkers:")
print("  X:", X_bio_train.shape)
print("  Mask:", M_bio_train.shape)

print("\nNutrition:")
print("  X:", X_nutri_train.shape)
print("  Mask:", M_nutri_train.shape)

print("\nLifestyle:")
print("  X:", X_life_train.shape)
print("  Mask:", M_life_train.shape)

print("\nDemographics:")
print("  X:", X_demo_train.shape)
print("  Mask:", M_demo_train.shape)


# ------------------------------------------------------------
# Hard checks
# ------------------------------------------------------------

assert X_bio_train.shape[1] == M_bio_train.shape[1] == len(BIOMARKER_COLS)

assert X_nutri_train.shape[1] == M_nutri_train.shape[1] == len(NUTRITION_COLS)

assert X_life_train.shape[1] == M_life_train.shape[1] == len(LIFESTYLE_COLS)

assert X_demo_train.shape[1] == M_demo_train.shape[1] == len(DEMO_COLS)


# Verify no NaNs remain after imputation
assert not np.isnan(X_bio_train).any()
assert not np.isnan(X_nutri_train).any()
assert not np.isnan(X_life_train).any()
assert not np.isnan(X_demo_train).any()


print("\nShape checks passed.")
print("Data tensors and missingness masks are aligned.")
print("No NaNs remain after preprocessing.")

Biomarkers:
  X: (29500, 8)
  Mask: (29500, 8)

Nutrition:
  X: (29500, 83)
  Mask: (29500, 83)

Lifestyle:
  X: (29500, 184)
  Mask: (29500, 184)

Demographics:
  X: (29500, 7)
  Mask: (29500, 7)

Shape checks passed.
Data tensors and missingness masks are aligned.
No NaNs remain after preprocessing.


In [28]:
# 9. Prepare multitask labels and masks

def prepare_targets(idx):
    y_raw = df.loc[idx, TARGET_COLS].to_numpy(dtype=np.float32)
    mask = (~np.isnan(y_raw)).astype(np.float32)
    y = np.nan_to_num(y_raw, nan=0.0).astype(np.float32)
    return y, mask

y_train, ym_train = prepare_targets(train_idx)
y_val, ym_val = prepare_targets(val_idx)
y_test, ym_test = prepare_targets(test_idx)

print("Training target availability:")
for i, c in enumerate(TARGET_COLS):
    print(c, int(ym_train[:, i].sum()), "/", len(y_train))

Training target availability:
TARGET_DIABETES 28841 / 29500
TARGET_OBESITY 27886 / 29500
TARGET_CVD 27839 / 29500


In [29]:
# 10. PyTorch dataset

class MultiModalDataset(Dataset):
    def __init__(
        self,
        bio_x, bio_m,
        nutri_x, nutri_m,
        life_x, life_m,
        demo_x, demo_m,
        y, y_mask
    ):
        self.bio_x = torch.tensor(bio_x, dtype=torch.float32)
        self.bio_m = torch.tensor(bio_m, dtype=torch.float32)
        self.nutri_x = torch.tensor(nutri_x, dtype=torch.float32)
        self.nutri_m = torch.tensor(nutri_m, dtype=torch.float32)
        self.life_x = torch.tensor(life_x, dtype=torch.float32)
        self.life_m = torch.tensor(life_m, dtype=torch.float32)
        self.demo_x = torch.tensor(demo_x, dtype=torch.float32)
        self.demo_m = torch.tensor(demo_m, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.y_mask = torch.tensor(y_mask, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "bio": self.bio_x[idx],
            "bio_missing": self.bio_m[idx],
            "nutri": self.nutri_x[idx],
            "nutri_missing": self.nutri_m[idx],
            "life": self.life_x[idx],
            "life_missing": self.life_m[idx],
            "demo": self.demo_x[idx],
            "demo_missing": self.demo_m[idx],
            "y": self.y[idx],
            "y_mask": self.y_mask[idx],
        }

train_ds = MultiModalDataset(
    X_bio_train, M_bio_train,
    X_nutri_train, M_nutri_train,
    X_life_train, M_life_train,
    X_demo_train, M_demo_train,
    y_train, ym_train
)

val_ds = MultiModalDataset(
    X_bio_val, M_bio_val,
    X_nutri_val, M_nutri_val,
    X_life_val, M_life_val,
    X_demo_val, M_demo_val,
    y_val, ym_val
)

test_ds = MultiModalDataset(
    X_bio_test, M_bio_test,
    X_nutri_test, M_nutri_test,
    X_life_test, M_life_test,
    X_demo_test, M_demo_test,
    y_test, ym_test
)

BATCH_SIZE = 256

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Loaders ready.")

Loaders ready.


# 11. Unified model

Each modality has a Transformer encoder. The outputs are latent representations created during the forward pass:

- `Z_bio`
- `Z_nutri`
- `Z_life`

They are **not separate pretrained files**.

All three are trained jointly inside one model.

In [30]:
# 11. Transformer modality encoder

class TabularTransformerEncoder(nn.Module):
    def __init__(self, n_features, d_model=128, nhead=8, layers=2, dropout=0.15):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        self.value_proj = nn.Linear(1, d_model)
        self.missing_proj = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(
            torch.randn(1, n_features, d_model) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, missing):
        # x: [B, F]
        # missing: [B, F]
        values = self.value_proj(x.unsqueeze(-1))
        miss = self.missing_proj(missing.unsqueeze(-1))

        tokens = values + miss + self.feature_embedding
        tokens = self.encoder(tokens)

        # Mean pooling creates one latent vector per patient.
        z = tokens.mean(dim=1)
        return self.norm(z)

In [31]:
# 12. Cross-modal Transformer

class CrossModalTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=8, layers=2, dropout=0.15):
        super().__init__()

        self.modality_embeddings = nn.Parameter(
            torch.randn(1, 3, d_model) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bio, z_nutri, z_life):
        tokens = torch.stack(
            [z_bio, z_nutri, z_life],
            dim=1
        )

        tokens = tokens + self.modality_embeddings
        tokens = self.encoder(tokens)

        z_fused = tokens.mean(dim=1)
        return self.norm(z_fused), tokens

In [32]:
# 13. Adaptive Mixture of Experts

class AdaptiveMoE(nn.Module):
    def __init__(self, d_model=128, n_experts=4, dropout=0.15):
        super().__init__()
        self.n_experts = n_experts

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 2, d_model),
                nn.LayerNorm(d_model)
            )
            for _ in range(n_experts)
        ])

        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, n_experts)
        )

    def forward(self, x):
        gate_logits = self.gate(x)
        weights = torch.softmax(gate_logits, dim=-1)

        expert_outputs = torch.stack(
            [expert(x) for expert in self.experts],
            dim=1
        )

        out = (expert_outputs * weights.unsqueeze(-1)).sum(dim=1)
        return out, weights

In [33]:
# 14. Complete BioNutriGPT model

class BioNutriGPT(nn.Module):
    def __init__(
        self,
        n_bio,
        n_nutri,
        n_life,
        n_demo,
        d_model=128,
        dropout=0.15
    ):
        super().__init__()

        self.bio_encoder = TabularTransformerEncoder(
            n_bio, d_model=d_model, dropout=dropout
        )

        self.nutri_encoder = TabularTransformerEncoder(
            n_nutri, d_model=d_model, dropout=dropout
        )

        self.life_encoder = TabularTransformerEncoder(
            n_life + n_demo,
            d_model=d_model,
            dropout=dropout
        )

        self.cross_modal = CrossModalTransformer(
            d_model=d_model,
            dropout=dropout
        )

        self.moe = AdaptiveMoE(
            d_model=d_model,
            n_experts=4,
            dropout=dropout
        )

        self.diabetes_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

        self.obesity_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

        self.cvd_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(
        self,
        bio, bio_missing,
        nutri, nutri_missing,
        life, life_missing,
        demo, demo_missing
    ):
        z_bio = self.bio_encoder(bio, bio_missing)
        z_nutri = self.nutri_encoder(nutri, nutri_missing)

        # Lifestyle + demographics form the third branch.
        context_x = torch.cat([life, demo], dim=1)
        context_missing = torch.cat([life_missing, demo_missing], dim=1)

        z_life = self.life_encoder(
            context_x,
            context_missing
        )

        z_fused, modality_tokens = self.cross_modal(
            z_bio,
            z_nutri,
            z_life
        )

        z_moe, gate_weights = self.moe(z_fused)

        logits = torch.cat([
            self.diabetes_head(z_moe),
            self.obesity_head(z_moe),
            self.cvd_head(z_moe)
        ], dim=1)

        return {
            "logits": logits,
            "z_bio": z_bio,
            "z_nutri": z_nutri,
            "z_life": z_life,
            "z_fused": z_fused,
            "z_moe": z_moe,
            "modality_tokens": modality_tokens,
            "gate_weights": gate_weights,
        }

In [34]:
# 15. Instantiate the ONE unified model

model = BioNutriGPT(
    n_bio=X_bio_train.shape[1],
    n_nutri=X_nutri_train.shape[1],
    n_life=X_life_train.shape[1],
    n_demo=X_demo_train.shape[1],
    d_model=128,
    dropout=0.15
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

BioNutriGPT(
  (bio_encoder): TabularTransformerEncoder(
    (value_proj): Linear(in_features=1, out_features=128, bias=True)
    (missing_proj): Linear(in_features=1, out_features=128, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.15, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.15, inplace=False)
          (dropout2): Dropout(p=0.15, inplace=False)
        )
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=Tru

In [35]:
# 16. One optimizer + multitask masked loss

# Compute task-specific positive weights from observed training labels.
pos_weights = []

for j in range(3):
    observed = y_train[ym_train[:, j] == 1, j]
    positives = observed.sum()
    negatives = len(observed) - positives

    if positives > 0:
        pw = negatives / positives
    else:
        pw = 1.0

    pos_weights.append(pw)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("Positive weights:", pos_weights.detach().cpu().numpy())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights,
    reduction="none"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

print("One optimizer configured for the entire model.")

Positive weights: [6.993625  1.6826359 8.851027 ]
One optimizer configured for the entire model.


In [36]:
# 17. Loss function with missing-target masking

def masked_multitask_loss(logits, y, y_mask):
    per_target = criterion(logits, y)

    masked = per_target * y_mask

    task_denominator = y_mask.sum(dim=0).clamp_min(1.0)
    task_losses = masked.sum(dim=0) / task_denominator

    total = task_losses.mean()

    return total, task_losses

In [37]:
# 18. Evaluation utilities

def collect_predictions(loader):
    model.eval()

    all_logits = []
    all_y = []
    all_mask = []

    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch["bio"].to(DEVICE, non_blocking=True),
                batch["bio_missing"].to(DEVICE, non_blocking=True),
                batch["nutri"].to(DEVICE, non_blocking=True),
                batch["nutri_missing"].to(DEVICE, non_blocking=True),
                batch["life"].to(DEVICE, non_blocking=True),
                batch["life_missing"].to(DEVICE, non_blocking=True),
                batch["demo"].to(DEVICE, non_blocking=True),
                batch["demo_missing"].to(DEVICE, non_blocking=True),
            )["logits"]

            all_logits.append(logits.cpu())
            all_y.append(batch["y"])
            all_mask.append(batch["y_mask"])

    return (
        torch.cat(all_logits).numpy(),
        torch.cat(all_y).numpy(),
        torch.cat(all_mask).numpy()
    )

def compute_metrics(logits, y, mask):
    probs = 1 / (1 + np.exp(-logits))

    metrics = {}

    for j, task in enumerate(TARGET_COLS):
        valid = mask[:, j] == 1

        if valid.sum() == 0:
            metrics[f"{task}_roc_auc"] = np.nan
            metrics[f"{task}_pr_auc"] = np.nan
            continue

        yy = y[valid, j]
        pp = probs[valid, j]

        if len(np.unique(yy)) < 2:
            metrics[f"{task}_roc_auc"] = np.nan
            metrics[f"{task}_pr_auc"] = np.nan
        else:
            metrics[f"{task}_roc_auc"] = roc_auc_score(yy, pp)
            metrics[f"{task}_pr_auc"] = average_precision_score(yy, pp)

    valid_aucs = [
        metrics[f"{t}_roc_auc"]
        for t in TARGET_COLS
        if not np.isnan(metrics[f"{t}_roc_auc"])
    ]

    metrics["mean_roc_auc"] = (
        float(np.mean(valid_aucs)) if valid_aucs else np.nan
    )

    return metrics

In [40]:
# ============================================================
# REINITIALIZE MODEL FROM SCRATCH
# ============================================================

# IMPORTANT:
# We are creating a completely fresh model after the previous
# NaN training attempt.

model = BioNutriGPT(
    n_bio=X_bio_train.shape[1],
    n_nutri=X_nutri_train.shape[1],
    n_life=X_life_train.shape[1],
    n_demo=X_demo_train.shape[1],
    d_model=128,
    dropout=0.15
).to(DEVICE)


# ------------------------------------------------------------
# Verify model parameters are finite
# ------------------------------------------------------------

bad_parameters = []

for name, param in model.named_parameters():

    if not torch.isfinite(param).all():
        bad_parameters.append(name)

if bad_parameters:
    raise RuntimeError(
        "Fresh model contains NaN/Inf parameters:\n"
        + "\n".join(bad_parameters)
    )

print("All fresh model parameters are finite.")


# ============================================================
# LOSS
# ============================================================

pos_weights = []

for j in range(3):

    observed = y_train[ym_train[:, j] == 1, j]

    positives = observed.sum()
    negatives = len(observed) - positives

    if positives > 0:
        pw = negatives / positives
    else:
        pw = 1.0

    # Safety limit
    pw = min(float(pw), 20.0)

    pos_weights.append(pw)


pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("Positive weights:", pos_weights.detach().cpu().numpy())


criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights,
    reduction="none"
)


# ============================================================
# NEW OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


print("Fresh model created.")
print("Fresh optimizer created.")

All fresh model parameters are finite.
Positive weights: [6.993625  1.6826359 8.851027 ]
Fresh model created.
Fresh optimizer created.


In [44]:
# ============================================================
# FORWARD-PASS DIAGNOSTIC
# ============================================================

model.eval()

batch = next(iter(train_loader))

with torch.no_grad():

    bio = batch["bio"].to(DEVICE)
    bio_missing = batch["bio_missing"].to(DEVICE)

    nutri = batch["nutri"].to(DEVICE)
    nutri_missing = batch["nutri_missing"].to(DEVICE)

    life = batch["life"].to(DEVICE)
    life_missing = batch["life_missing"].to(DEVICE)

    demo = batch["demo"].to(DEVICE)
    demo_missing = batch["demo_missing"].to(DEVICE)

    # Check inputs
    print("Input checks:")
    print("Bio:", torch.isfinite(bio).all().item())
    print("Nutrition:", torch.isfinite(nutri).all().item())
    print("Lifestyle:", torch.isfinite(life).all().item())
    print("Demographics:", torch.isfinite(demo).all().item())

    # Individual branches
    z_bio = model.bio_encoder(
        bio,
        bio_missing
    )

    print("\nZ_bio finite:", torch.isfinite(z_bio).all().item())

    z_nutri = model.nutri_encoder(
        nutri,
        nutri_missing
    )

    print("Z_nutri finite:", torch.isfinite(z_nutri).all().item())

    context_x = torch.cat(
        [life, demo],
        dim=1
    )

    context_missing = torch.cat(
        [life_missing, demo_missing],
        dim=1
    )

    z_life = model.life_encoder(
        context_x,
        context_missing
    )

    print("Z_life finite:", torch.isfinite(z_life).all().item())

    # Cross-modal attention
    z_fused, modality_tokens = model.cross_modal(
        z_bio,
        z_nutri,
        z_life
    )

    print("Z_fused finite:", torch.isfinite(z_fused).all().item())

    # MoE
    z_moe, gate_weights = model.moe(z_fused)

    print("Z_MoE finite:", torch.isfinite(z_moe).all().item())
    print("Gate weights finite:", torch.isfinite(gate_weights).all().item())

    # Final heads
    logits = torch.cat([
        model.diabetes_head(z_moe),
        model.obesity_head(z_moe),
        model.cvd_head(z_moe)
    ], dim=1)

    print("Final logits finite:", torch.isfinite(logits).all().item())

    print("\nLogits:")
    print(logits[:5])

Input checks:
Bio: True
Nutrition: True
Lifestyle: True
Demographics: True

Z_bio finite: True
Z_nutri finite: True
Z_life finite: True
Z_fused finite: True
Z_MoE finite: True
Gate weights finite: True
Final logits finite: True

Logits:
tensor([[-4.2279, -0.5965, -3.4668],
        [ 1.3091,  0.0547,  1.6915],
        [-0.2196,  1.4628, -0.6112],
        [-1.3665, -0.3244,  0.0711],
        [-2.9589, -0.2140, -1.5205]], device='cuda:0')


In [41]:
# 19. STABLE END-TO-END TRAINING LOOP
#
# First train in FP32 for numerical stability.
# Once this is confirmed stable, AMP can be re-enabled.

EPOCHS = 20
best_val_auc = -np.inf
history = []

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_loss = 0.0
    n_batches = 0

    for batch_idx, batch in enumerate(train_loader):

        optimizer.zero_grad(set_to_none=True)

        # Move inputs to GPU
        bio = batch["bio"].to(DEVICE, non_blocking=True)
        bio_missing = batch["bio_missing"].to(DEVICE, non_blocking=True)

        nutri = batch["nutri"].to(DEVICE, non_blocking=True)
        nutri_missing = batch["nutri_missing"].to(DEVICE, non_blocking=True)

        life = batch["life"].to(DEVICE, non_blocking=True)
        life_missing = batch["life_missing"].to(DEVICE, non_blocking=True)

        demo = batch["demo"].to(DEVICE, non_blocking=True)
        demo_missing = batch["demo_missing"].to(DEVICE, non_blocking=True)

        y = batch["y"].to(DEVICE, non_blocking=True)
        y_mask = batch["y_mask"].to(DEVICE, non_blocking=True)

        # Safety checks
        if not torch.isfinite(bio).all():
            raise RuntimeError(f"NaN/Inf found in bio at batch {batch_idx}")

        if not torch.isfinite(nutri).all():
            raise RuntimeError(f"NaN/Inf found in nutrition at batch {batch_idx}")

        if not torch.isfinite(life).all():
            raise RuntimeError(f"NaN/Inf found in lifestyle at batch {batch_idx}")

        if not torch.isfinite(demo).all():
            raise RuntimeError(f"NaN/Inf found in demographics at batch {batch_idx}")

        # ---------------------------------------------------------
        # Forward pass
        # FP32 intentionally used here for stability.
        # ---------------------------------------------------------

        out = model(
            bio,
            bio_missing,
            nutri,
            nutri_missing,
            life,
            life_missing,
            demo,
            demo_missing
        )

        logits = out["logits"]

        # Check model output
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                f"Model produced NaN/Inf logits at "
                f"epoch {epoch}, batch {batch_idx}"
            )

        # ---------------------------------------------------------
        # Joint multitask loss
        # ---------------------------------------------------------

        loss, task_losses = masked_multitask_loss(
            logits,
            y,
            y_mask
        )

        if not torch.isfinite(loss):
            print("Invalid loss detected.")
            print("Task losses:", task_losses)
            print("Logits min:", logits.min().item())
            print("Logits max:", logits.max().item())
            raise RuntimeError(
                f"NaN/Inf loss at epoch {epoch}, batch {batch_idx}"
            )

        # ---------------------------------------------------------
        # Backpropagation through the ENTIRE unified model
        # ---------------------------------------------------------

        loss.backward()

        # Gradient clipping for Transformer stability
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    train_loss = running_loss / max(n_batches, 1)

    # -------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------

    val_logits, val_y_np, val_mask_np = collect_predictions(
        val_loader
    )

    # Make sure validation predictions are finite
    if not np.isfinite(val_logits).all():
        raise RuntimeError(
            f"Validation predictions contain NaN/Inf at epoch {epoch}"
        )

    val_metrics = compute_metrics(
        val_logits,
        val_y_np,
        val_mask_np
    )

    val_auc = val_metrics["mean_roc_auc"]

    if np.isnan(val_auc):
        print("Warning: validation AUC is NaN.")
        val_auc_for_scheduler = 0.0
    else:
        val_auc_for_scheduler = val_auc

    scheduler.step(val_auc_for_scheduler)

    record = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        **{
            k: float(v) if isinstance(v, (float, np.floating)) else v
            for k, v in val_metrics.items()
        }
    }

    history.append(record)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"val_mean_auc={val_auc:.4f}"
    )

    # -------------------------------------------------------------
    # Save best model
    # -------------------------------------------------------------

    if not np.isnan(val_auc) and val_auc > best_val_auc:

        best_val_auc = val_auc

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_auc": best_val_auc,
                "biomarker_cols": BIOMARKER_COLS,
                "nutrition_cols": NUTRITION_COLS,
                "lifestyle_cols": LIFESTYLE_COLS,
                "demo_cols": DEMO_COLS,
                "target_cols": TARGET_COLS,
            },
            OUTPUT_DIR / "best_bionutrigpt_no_liver.pt"
        )

        print("  -> saved best checkpoint")

Epoch 01/20 | loss=1.0618 | val_mean_auc=0.6631
  -> saved best checkpoint
Epoch 02/20 | loss=1.0117 | val_mean_auc=0.6852
  -> saved best checkpoint
Epoch 03/20 | loss=0.9785 | val_mean_auc=0.7091
  -> saved best checkpoint
Epoch 04/20 | loss=0.9167 | val_mean_auc=0.7519
  -> saved best checkpoint
Epoch 05/20 | loss=0.8802 | val_mean_auc=0.7605
  -> saved best checkpoint
Epoch 06/20 | loss=0.8688 | val_mean_auc=0.7714
  -> saved best checkpoint
Epoch 07/20 | loss=0.8275 | val_mean_auc=0.8041
  -> saved best checkpoint
Epoch 08/20 | loss=0.7913 | val_mean_auc=0.8125
  -> saved best checkpoint
Epoch 09/20 | loss=0.7835 | val_mean_auc=0.8164
  -> saved best checkpoint
Epoch 10/20 | loss=0.7758 | val_mean_auc=0.8193
  -> saved best checkpoint
Epoch 11/20 | loss=0.7681 | val_mean_auc=0.8208
  -> saved best checkpoint
Epoch 12/20 | loss=0.7712 | val_mean_auc=0.8209
  -> saved best checkpoint
Epoch 13/20 | loss=0.7690 | val_mean_auc=0.8228
  -> saved best checkpoint
Epoch 14/20 | loss=0.7594

In [42]:
# 20. Final test evaluation

checkpoint_path = OUTPUT_DIR / "best_bionutrigpt_no_liver.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

test_logits, test_y_np, test_mask_np = collect_predictions(test_loader)
test_metrics = compute_metrics(
    test_logits,
    test_y_np,
    test_mask_np
)

print("\nTest metrics")
print("=" * 50)

for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


Test metrics
TARGET_DIABETES_roc_auc: 0.9245
TARGET_DIABETES_pr_auc: 0.7417
TARGET_OBESITY_roc_auc: 0.7124
TARGET_OBESITY_pr_auc: 0.5839
TARGET_CVD_roc_auc: 0.8527
TARGET_CVD_pr_auc: 0.3670
mean_roc_auc: 0.8298


In [43]:
# 21. Save preprocessing metadata and training history

preprocessing = {
    "data_path": DATA_PATH,
    "biomarker_cols": BIOMARKER_COLS,
    "nutrition_cols": NUTRITION_COLS,
    "lifestyle_cols": LIFESTYLE_COLS,
    "demo_cols": DEMO_COLS,
    "target_cols": TARGET_COLS,
    "architecture": {
        "bio": "Tabular Transformer",
        "nutrition": "Tabular Transformer",
        "lifestyle_demographics": "Tabular Transformer",
        "fusion": "Cross-Modal Transformer",
        "head": "Adaptive Mixture of Experts",
        "tasks": TARGET_COLS,
    },
    "liver_attributes_used": False,
}

with open(OUTPUT_DIR / "preprocessing_metadata.json", "w") as f:
    json.dump(preprocessing, f, indent=2)

with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

joblib.dump(
    {
        "bio_imputer": bio_imp,
        "bio_scaler": bio_scaler,
        "nutri_imputer": nutri_imp,
        "nutri_scaler": nutri_scaler,
        "life_imputer": life_imp,
        "life_scaler": life_scaler,
        "demo_imputer": demo_imp,
        "demo_scaler": demo_scaler,
    },
    OUTPUT_DIR / "preprocessors.joblib"
)

print("Saved outputs to:", OUTPUT_DIR)

Saved outputs to: /content/bionutrigpt_multimodal_no_liver


## 22. End-to-end training confirmation

The training performed above is a **single optimization process**.

For every batch:

```text
final task loss
      ↓
Diabetes / Obesity / CVD heads
      ↓
Adaptive MoE
      ↓
Cross-Modal Transformer
      ↓
Z_bio / Z_nutri / Z_life
      ↓
three modality Transformers
      ↓
all trainable parameters
```

There is no separate pretraining stage and no requirement for saved `Z_bio`, `Z_nutri`, or `Z_life` files.

SyntaxError: invalid syntax (886743233.py, line 1)